### Install and import packages

In [1]:
import sys
!{sys.executable} -m pip install sqlalchemy pymysql pandas

import pandas as pd
import pymysql
import getpass # May need to import as `from getpass import getpass`
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Enter password and set database name


In [2]:
password = getpass.getpass("Enter your MySQL root password: ")
db_name = "movies"


Enter your MySQL root password:  ········


### Create the movies database

In [3]:
conn = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=password
)
cursor = conn.cursor()
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {db_name}")
conn.commit()
cursor.close()
conn.close()
print(f"Database `{db_name}` is ready.")

OperationalError: (2003, "Can't connect to MySQL server on '127.0.0.1' ([WinError 10061] No connection could be made because the target machine actively refused it)")

### Connect to the movies database

In [ ]:
url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password=password,
    host="127.0.0.1",
    port=3306,
    database=db_name,
)

engine = create_engine(url)

with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE();"))
    print("Connected to database:", result.scalar())

### Create all 6 tables

In [ ]:
with engine.connect() as connection:
    connection.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    connection.execute(text("DROP TABLE IF EXISTS production_classification;"))
    connection.execute(text("DROP TABLE IF EXISTS content_description;"))
    connection.execute(text("DROP TABLE IF EXISTS media_resources;"))
    connection.execute(text("DROP TABLE IF EXISTS financial_metrics;"))
    connection.execute(text("DROP TABLE IF EXISTS popularity_metrics;"))
    connection.execute(text("DROP TABLE IF EXISTS general_information;"))
    connection.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

    connection.execute(text("""
        CREATE TABLE general_information (
            id INT PRIMARY KEY,
            title VARCHAR(255),
            original_title VARCHAR(255),
            status VARCHAR(50),
            release_date DATE,
            runtime INT,
            adult BOOLEAN,
            original_language VARCHAR(10)
        );
    """))

    connection.execute(text("""
        CREATE TABLE popularity_metrics (
            id INT PRIMARY KEY,
            vote_average FLOAT,
            vote_count INT,
            popularity FLOAT,
            FOREIGN KEY (id) REFERENCES general_information(id)
        );
    """))

    connection.execute(text("""
        CREATE TABLE financial_metrics (
            id INT PRIMARY KEY,
            budget INT,
            revenue INT,
            FOREIGN KEY (id) REFERENCES general_information(id)
        );
    """))

    connection.execute(text("""
        CREATE TABLE media_resources (
            id INT PRIMARY KEY,
            homepage VARCHAR(500),
            imdb_id VARCHAR(20),
            poster_path VARCHAR(255),
            backdrop_path VARCHAR(255),
            FOREIGN KEY (id) REFERENCES general_information(id)
        );
    """))

    connection.execute(text("""
        CREATE TABLE content_description (
            id INT PRIMARY KEY,
            overview TEXT,
            tagline VARCHAR(500),
            keywords TEXT,
            FOREIGN KEY (id) REFERENCES general_information(id)
        );
    """))

    connection.execute(text("""
        CREATE TABLE production_classification (
            id INT PRIMARY KEY,
            genres TEXT,
            production_companies TEXT,
            production_countries TEXT,
            spoken_languages TEXT,
            FOREIGN KEY (id) REFERENCES general_information(id)
        );
    """))

    connection.commit()

print("All 6 tables created.")

### Load the CSV

In [ ]:
csv_path = "/Users/abigailwesterlund/Documents/Forge/TheSQLSequel/TMDB_movie_dataset_25k_samples.csv"

df = pd.read_csv(csv_path)
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce').dt.strftime('%Y-%m-%d')
df['adult'] = df['adult'].apply(lambda x: 1 if x is True else 0)
df = df.where(pd.notnull(df), None)

print(f"Loaded {len(df)} rows from CSV.")
df.head(3)

### Insert data into tables

In [ ]:
df[['id','title','original_title','status','release_date','runtime','adult','original_language']] \
    .to_sql('general_information', con=engine, if_exists='append', index=False)

df[['id','vote_average','vote_count','popularity']] \
    .to_sql('popularity_metrics', con=engine, if_exists='append', index=False)

df[['id','budget','revenue']] \
    .to_sql('financial_metrics', con=engine, if_exists='append', index=False)

df[['id','homepage','imdb_id','poster_path','backdrop_path']] \
    .to_sql('media_resources', con=engine, if_exists='append', index=False)

df[['id','overview','tagline','keywords']] \
    .to_sql('content_description', con=engine, if_exists='append', index=False)

df[['id','genres','production_companies','production_countries','spoken_languages']] \
    .to_sql('production_classification', con=engine, if_exists='append', index=False)

print("All data inserted.")

### Check row counts

In [ ]:
tables = ['general_information','popularity_metrics','financial_metrics',
          'media_resources','content_description','production_classification']

for table in tables:
    count = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table};", con=engine)
    print(f"{table}: {count['n'][0]} rows")